In [ ]:
using Bessels
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/simulator.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_settings.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/visualization/plot_utils.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ε_p = 0.85
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d= 10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4;

In [ ]:
g2/2pi*1e3

# Check for the Mean Field

In [ ]:
t_max=45/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ε_d, ω_d, buffer_phase, ε_p, ω_p, pump_phase)
environment = MeanField(0.0, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment)

In [ ]:
ϕ_rotating_a =@. ϕ_CL_a * cos(ω_a * t_list_CL) + q_CL_a*Z_a * sin(-ω_a * t_list_CL)
q_rotating_a =@. -ϕ_CL_a/Z_a * sin(-ω_a * t_list_CL) + q_CL_a * cos(ω_a * t_list_CL)

Re_rotating_a = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a = q_rotating_a * sqrt(Z_a/2)

ϕ_rotating_b =@. ϕ_CL_b * cos(ω_b * t_list_CL) + q_CL_b*Z_b * sin(-ω_b * t_list_CL)
q_rotating_b =@. -ϕ_CL_b/Z_b * sin(-ω_b * t_list_CL) + q_CL_b * cos(ω_b * t_list_CL)

Re_rotating_b = ϕ_rotating_b / sqrt(2*Z_b)
Im_rotating_b = q_rotating_b * sqrt(Z_b/2)

plot(Re_rotating_a[1:10:end], Im_rotating_a[1:10:end])


In [ ]:

using JSON
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/limit_cycles/unstable_trajectory_memory_re_09_04_2026.json", "w") do io
    JSON.print(io, Re_rotating_a[1:10:end])
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/limit_cycles/unstable_trajectory_memory_im_09_04_2026.json", "w") do io
    JSON.print(io, Im_rotating_a[1:10:end])
end

using JSON
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/limit_cycles/unstable_trajectory_buffer_re_09_04_2026.json", "w") do io
    JSON.print(io, Re_rotating_b[1:10:end])
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/limit_cycles/unstable_trajectory_buffer_im_09_04_2026.json", "w") do io
    JSON.print(io, Im_rotating_b[1:10:end])
end

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a[1:10:end], q_CL_a[1:10:end], t_list_CL[1:10:end], ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)